In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, resnet152, regnet_y_800mf, mobilenet_v3_large, efficientnet_v2_l

from models.yolo1 import YOLOv1


In [2]:
YOLOv1(num_cl=5, backbone=None, bn=False, act='relu')

YOLOv1(
  (fn_act): ModuleDict(
    (relu): ReLU(inplace=True)
    (leakyrelu): LeakyReLU(negative_slope=0.01, inplace=True)
  )
  (backbone): BackBone(
    (layers): Sequential(
      (0): ConvBA(
        (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (fn_act): ModuleDict(
          (relu): ReLU(inplace=True)
          (leakyrelu): LeakyReLU(negative_slope=0.01, inplace=True)
        )
      )
      (1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
      (2): ConvBA(
        (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (fn_act): ModuleDict(
          (relu): ReLU(inplace=True)
          (leakyrelu): LeakyReLU(negative_slope=0.01, inplace=True)
        )
      )
      (3): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ConvBlock(
        (conv1x1): ConvBA(
          (conv): Conv2d(192, 128, kernel_size=(1, 1), stride=(1, 1))
          (f

In [3]:
rn18 = resnet18(weights="DEFAULT")
rn152 = resnet152(weights="DEFAULT")
regnet_y = regnet_y_800mf(weights="DEFAULT")
mobilenet = mobilenet_v3_large(weights="DEFAULT")
efficientnet = efficientnet_v2_l(weights="DEFAULT")

backbone_18 = nn.Sequential(
    rn18.conv1,
    rn18.bn1,
    rn18.relu,
    rn18.maxpool,
    rn18.layer1,
    rn18.layer2,
    rn18.layer3,
    rn18.layer4
)

backbone_152 = nn.Sequential(
    rn152.conv1,
    rn152.bn1,
    rn152.relu,
    rn152.maxpool,
    rn152.layer1,
    rn152.layer2,
    rn152.layer3,
    rn152.layer4
)

backbone_r = nn.Sequential(
    regnet_y.stem,
    regnet_y.trunk_output
)

backbone_m = mobilenet.features

backbone_e = efficientnet.features

In [4]:
inp = torch.rand(1, 3, 448, 448)

model = YOLOv1(
    num_cl=20,
    backbone=backbone_e,
    bn=True,
    act='relu',
    use_layer="conv3x3",
    use_sigmoid="full_sigmoid"
)

model.train()
pred = model(inp)


In [5]:
inp = torch.rand(1, 3, 448, 448)

model = YOLOv1(
    num_cl=20,
    backbone=backbone_e,
    bn=True,
    act='relu',
    use_layer="conv3x3",
    use_sigmoid="full_sigmoid"
)

model.eval()
pred = model(inp)

In [6]:
import os 
import random
import torch
import numpy as np
import torch.nn as nn
import torchvision
from pathlib import Path
from torch.optim import SGD
from torch.optim.lr_scheduler import MultiStepLR
from models.yolo1 import YOLOv1
from utils.dataloaders import create_loader
from utils.nms import nms_yolov1
from utils.general import clip_boxes, process_batch
from loss.loss import YOLOv1Loss
from metrics.metrics import ap_per_class
from tqdm import tqdm

ROOT = Path().resolve()
EPOCHS = 135
BATCH_SIZE = 64
seed = 1961
threshold = 0.01

freez_parameters = False
save_dir = ROOT.joinpath("save_model")

r34 = torchvision.models.resnet34(
    weights='DEFAULT',
)

backbone = nn.Sequential(
    r34.conv1,
    r34.bn1,
    r34.relu,
    r34.maxpool,
    r34.layer1,
    r34.layer2,
    r34.layer3,
    r34.layer4,
)

print(f"os_cpu: {os.cpu_count()}")
print(ROOT)

os_cpu: 6
H:\Programming\Machine_learning\deep machine learning\VIDEO_CURSE\Education_localization


In [7]:
np.random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("Подготовка данных")
train_loader, val_loader = create_loader(
    [
        ROOT.parent.joinpath("data/VOCdevkit/VOC2012"),
        ROOT.parent.joinpath("data/VOCdevkit/VOC2007"),
        ROOT.parent.joinpath("data/VOCdevkit/VOC22"),
    ],
    split=[0.8, 0.2],
    batch_size=BATCH_SIZE,
    workers=1,
    pin_memory=True,
    shuffle_train=True
)

print("Подготовка ")

model = YOLOv1(
    backbone=backbone,
    use_layer="conv1x1",
    use_sigmoid="not_full_sigmoid"
)

if freez_parameters:
    for param in model.backbone.parameters():
        param.requires_grad=False

loss_model = YOLOv1Loss()
opt = SGD(
    params=model.parameters(),
    lr=0.001,
    weight_decay=5e-4,
    momentum=0.9
)

lr_scheduler = MultiStepLR(
    opt,
    milestones=[75, 105],
    gamma=0.5
)

if not save_dir.exists():
    os.mkdir(save_dir)

Подготовка данных
В наборе данных 22136 размеченных изображений.
Подготовка 


In [ ]:
train_loss, val_loss, lr_list, metrics, best_loss = [], [], [], [], None
print("Старт цикла обучения")

for epoch in range(EPOCHS):
    model.train()
    loss_f = []
    train_loop = tqdm(train_loader, leave=False)
    for img, targets, _ in train_loop:

        preds = model(img)
        loss, *loss_list = loss_model(preds, targets)
        opt.zero_grad()
        loss.backward()
        opt.step()

        loss_f.append(loss_list)
        loss_list = np.array(loss_f).mean(axis=0)

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], trian_loss={loss_list[0]:.4f}, x_loss={loss_list[1]:.4f}...")

    train_loss.append(loss_list[0])

    model.eval()
    with torch.no_grad():
        loss_f = []
        stats = []

        iouv = torch.linspace(0.5, 0.95, 10)
        niou = iouv.numel()

        for img, targets, boxes_classes in val_loader:

            preds, boxes_conf_class = model(img)
            loss, *loss_list = loss_model(preds, targets)

            preds_nms = nms_yolov1(boxes_conf_class, 0.25, 0.45)

            for idx, pred in enumerate(preds_nms):
                labels = boxes_conf_class[idx]
                nl, npr = labels.shape[0], pred.shape[0]
                correct = torch.zeros(npr, niou, dtype=torch.bool)

                if npr == 0:
                    if nl:
                        stats.append((correct, *torch.zeros((2, 0)), labels[:, 4]))
                    continue
                predn = pred.clone()

                if nl:
                    tbox = labels[:, :4]
                    clip_boxes(tbox)
                    labelsn = torch.cat((labels[:, 4:], tbox), 1)
                    correct = process_batch(predn, labelsn, iouv)

                stats.append((correct, pred[:, 4], pred[:, 5], labels[:, 4]))

            loss_f.append(loss_list)

    stats = [torch.cat(x, 0).cpu().numpy() for x in zip(*stats)]
    if len(stats) and stats[0].any():
        ap = ap_per_class(*stats)
        ap50, ap = ap[:, 0], ap.mean(1)
        map50, mAP = ap50.mean(), ap.mean()
        metrics.append((map50, mAP))

        print(f"Epoch [{epoch+1}/{EPOCHS}]: mAP@0.5 = {map50:.4f}, map@0.5:0.95={mAP:.4f}")

    else:
        print(f"Epoch [{epoch +1}/{EPOCHS}]: метрика не рассчитивалась")


Старт цикла обучения


  0%|          | 0/277 [00:00<?, ?it/s]